In [118]:
athletes = {
    "Athlete12": {
        "sport": "rowing",
        "ftp": 230,
        "fit_dir": "AthleteDataCoding/Athlete12/OGs",
        "label_dir": "AthleteDataCoding/Athlete12/GTs",
        "allowed_files": {
            "10536403349_5006_row.fit",
            "10543962115_10006_row.fit",
            "10551999765_10006_row.fit",
            "10652950510_Btchen_fahren.fit",
            "10674304801_Btchen_fahren_in_Etappen_3.fit",
            "10694767945_Btchen_fahren_.fit",
            "10809067165_3007_row.fit",
            "11636429453_4558_row.fit",
            "11783093951_4x2000_sub8.fit",
            "11791568584_.fit",
            "11808467517_.fit",
            "11838948742_3006_row.fit",
            "11846980624_Platt_.fit",
            "11855866225_In_den_Seilen.fit",
            "11864381887_Catwalk_.fit",
            "11912062341_500er_in_grau.fit",
            "11962243206_Wundmanagement.fit",
            "11971395278_Use_it_or_lose_it.fit",
            "11987690514_Besser_als_Nix.fit",
            "11994450315_Airobics.fit",
            "12036692734_Exhausted.fit",
            "12069656901_Schwitzen_im_Sitzen.fit",
            "12501679452_Zustand_nach_Xtem_Atemwegsinfekt.fit",
            "12806981726_Row_Stretch__Stabi.fit",
            "12846436186_Synchronflug.fit",
            "12927701413_I_have_no_idea_when_Ill_be_back_in_serious_training_routine_but_at_least_we_have_a_kitchen.fit",
            "12951604563_DienstSport.fit",
            "13010348229_1h_w_4x1_intensity.fit",
            "13039020832_Analytiker.fit",
            "13280559542_Warm_up_rowing.fit",
            "13300350440_W_Up.fit",
            "13363035398_SGAktiv.fit",
            "13582048984_W_Up.fit",
            "13583093636_Afterburner_.fit",
            "13601462878_Zehnbauer.fit",
            "13609970768_Uffwrme.fit",
            "13610691264_1x_Crescendo.fit",
            "13618782252_3x5.fit",
            "13643807487_Nachfitten.fit",
            "13662882990_Heldentod.fit",
            "13672121049_Base_Miles.fit",
            "13688068283_Luftpresser.fit",
            "13918354210_W_Upen.fit",
            "13957096402_Technik.fit",
            "13971240869_A_ella_le_gusta.fit",
            "13974345688_Nochmaaaal.fit",
            "13983533934_Technik__30er.fit",
            "14001095362_Wer_will_der_kann_.fit",
            "14038989670__Hyperthermie_.fit",
            "14077735636_Base.fit",
            "14089880174_Zn_IKEA.fit",
            "14114545767_Dampfnudel.fit",
            "14125110656_Vallah_isch_balla.fit",
            "14135321532_Pimp_my_ride.fit",
            "14156450361_On_a_mission.fit",
            "14174927764_Dunstabzugshaubenselfie.fit",
            "14182817844_The_Emptiness_Machine.fit",
            "14260930602_3x1010_SubThr.fit",
            "14313279747_Vernunft_verliert_.fit",
            "14374019349_Uff.fit",
            "14396237986_4659_row.fit"
        },
        "quarterly_metrics": {
            "2025 Q2": {"ftp": 269, "max_hr": 183, "rhr": 40},
            "2025 Q1": {"ftp": 282, "max_hr": 193, "rhr": 40},
            "2024 Q4": {"ftp": 292, "max_hr": 191, "rhr": 40},
            "2024 Q3": {"ftp": 306, "max_hr": 189, "rhr": 40},
            "2024 Q2": {"ftp": 294, "max_hr": 188, "rhr": 40},
            "2024 Q1": {"ftp": 298, "max_hr": 192, "rhr": 40},
        }
    },
    "Athlete2": {
        "sport": "biking",
        "ftp": 341,
        "fit_dir": "AthleteDataCoding/Athlete2/OGs",
        "label_dir": "AthleteDataCoding/Athlete2/GTs",
        "allowed_files": "all",
        "quarterly_metrics": {
            "2025 Q2": {"ftp": 345, "max_hr": 194, "rhr": None},
            "2025 Q1": {"ftp": 341, "max_hr": 193, "rhr": None},
            "2024 Q4": {"ftp": 308, "max_hr": 194, "rhr": None},
            "2024 Q3": {"ftp": 306, "max_hr": 191, "rhr": None},
            "2024 Q2": {"ftp": 306, "max_hr": 191, "rhr": None},
            "2024 Q1": {"ftp": 306, "max_hr": 191, "rhr": None},
        }
    }
}

In [119]:
import pandas as pd
import numpy as np
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [120]:
def create_hr_features(csv_path, athlete_name, athletes, window_size: int = 5,
                       long_window: int = 10, seven_zone_model: bool = True):
    """
    Create heart rate features from a CSV file with time series data.
    Automatically uses seasonal metrics (ftp, max_hr, rest_hr) based on timestamp.

    Returns:
        pd.DataFrame: DataFrame with new HR-related features (timestamp as column).
    """
    df = pd.read_csv(csv_path).copy()

    # Parse and sort timestamps
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp').drop_duplicates(subset='timestamp').reset_index(drop=True)

    # Reindex to fill missing seconds
    expected = pd.date_range(start=df['timestamp'].min(), end=df['timestamp'].max(), freq='1s')
    df = df.set_index('timestamp').reindex(expected).reset_index().rename(columns={'index': 'timestamp'})

    # Compute delta_time from timestamp column
    df['delta_time'] = df['timestamp'].diff().dt.total_seconds().fillna(0)

    # Binary manual labels
    df['manual_timestamp_numerical'] = df['Manual_Timestamps'].astype(str).str.lower().isin(['true', '1']).astype(int)
    df.loc[0, 'manual_timestamp_numerical'] = 1

    # Get seasonal values or fallback to base
    session_start = df['timestamp'].min()
    quarter = f"{session_start.year} Q{((session_start.month - 1) // 3) + 1}"
    athlete_meta = athletes[athlete_name]
    seasonal = athlete_meta.get("quarterly_metrics", {}).get(quarter, {})

    rest_hr = seasonal.get("rhr", None)
    max_hr = seasonal.get("max_hr", None)
    ftp = seasonal.get("ftp", athlete_meta["ftp"])

    # Rolling averages and derivatives
    df['power_roll_avg'] = df['power'].rolling(window=window_size, center=True, min_periods=1).mean()
    df['hr_roll_avg'] = df['heart_rate'].rolling(window=window_size, center=True, min_periods=1).mean()
    df['hr_derivative'] = df['hr_roll_avg'].diff() / df['delta_time']
    df['hr_second_derivative'] = df['hr_derivative'].diff() / df['delta_time']
    roll_avg_long = df['heart_rate'].rolling(window=long_window, center=True, min_periods=1).mean()
    df['hr_derivative_10s'] = roll_avg_long.diff() / df['delta_time']

    # HR zones
    if rest_hr is not None and max_hr is not None:
        intensity = (df['hr_roll_avg'] - rest_hr) / (max_hr - rest_hr)
        df['hr_zone_hr_reserve'] = pd.cut(intensity, bins=[-np.inf, 0.5, 0.7, 0.8, 0.9, np.inf], labels=[1, 2, 3, 4, 5]).astype('Int64')
    if max_hr is not None:
        df['hr_zone_olympia'] = pd.cut(df['hr_roll_avg'], bins=[-np.inf, 72, 82, 87, 92, np.inf], labels=[1, 2, 3, 4, 5]).astype('Int64')

    # Power zone classification
    if seven_zone_model:
        power_bins = [0, 55, 75, 90, 105, 120, 150, float('inf')]
        power_labels = list(range(1, 8))
    else:
        power_bins = [0, 55, 75, 90, 105, 120, float('inf')]
        power_labels = list(range(1, 7))

    df['Power_Zone'] = pd.cut(df['power_roll_avg'] / ftp * 100, bins=power_bins, labels=power_labels)
    df['Power_Zone'] = df['Power_Zone'].cat.codes.replace(-1, np.nan)
    df["Power_Zone_Label"] = df["Power_Zone"].map({i: f"Zone{i+1}" for i in range(7)})

    # Assign gt_zone_type using Power_Zone_Label
    df["gt_zone_type"] = "Unclassified"
    gt_times = df[df["manual_timestamp_numerical"] == 1]['timestamp'].to_list()

    if gt_times:
        for i in range(len(gt_times)):
            start = gt_times[i]
            end = gt_times[i + 1] if i + 1 < len(gt_times) else df['timestamp'].iloc[-1]
            mask = (df['timestamp'] >= start) & (df['timestamp'] <= end)
            zone_subset = df.loc[mask, "Power_Zone_Label"].dropna()
            dominant_zone = zone_subset.mode().iloc[0] if not zone_subset.empty else "Unclassified"
            df.loc[mask, "gt_zone_type"] = dominant_zone

    return df

In [121]:
def process_hr_features_for_all_sessions(athletes: dict) -> dict:
    session_data = {}

    for athlete, meta in athletes.items():
        label_dir = meta["label_dir"]
        allowed = meta["allowed_files"]

        if allowed == "all":
            allowed_csvs = {f for f in os.listdir(label_dir) if f.endswith("_with_manual_labels.csv")}
        else:
            allowed_csvs = {f.replace(".fit", "_with_manual_labels.csv") for f in allowed}

        for filename in allowed_csvs:
            file_path = os.path.join(label_dir, filename)
            if not os.path.exists(file_path):
                print(f"⚠️ Skipping missing file: {file_path}")
                continue

            try:
                df = create_hr_features(file_path, athlete_name=athlete, athletes=athletes)
                session_name = filename.replace("_with_manual_labels.csv", "")
                session_data[(athlete, session_name)] = df
            except Exception as e:
                print(f"❌ Error in {file_path}: {e}")

    return session_data


In [122]:
hr_session_data = process_hr_features_for_all_sessions(athletes)

In [123]:
# Pick a random session
example_key = list(hr_session_data.keys())[0]

# Print the key (athlete, session)
print("Inspecting session:", example_key)

# Print the first few rows
print(hr_session_data[example_key][['manual_timestamp_numerical', 'Power_Zone', 'Power_Zone_Label']].head(20))



Inspecting session: ('Athlete12', '10674304801_Btchen_fahren_in_Etappen_3')
    manual_timestamp_numerical  Power_Zone Power_Zone_Label
0                            1         0.0            Zone1
1                            0         0.0            Zone1
2                            0         0.0            Zone1
3                            0         0.0            Zone1
4                            0         0.0            Zone1
5                            0         0.0            Zone1
6                            0         0.0            Zone1
7                            0         0.0            Zone1
8                            0         0.0            Zone1
9                            0         0.0            Zone1
10                           0         0.0            Zone1
11                           0         0.0            Zone1
12                           0         0.0            Zone1
13                           0         0.0            Zone1
14                      

In [124]:
# 🎯 Fixed test sessions for consistent model evaluation (were picked randomly)
test_sessions = [
    ("Athlete2", "13363782092_Zwift__Aerobic_Mixup_in_New_York"),
    ("Athlete2", "i65696340_Zwift__LC16_Lactate_Clearance_Over_Under_in_Watopia"),
    ("Athlete12", "13601462878_Zehnbauer"),
    ("Athlete12", "12036692734_Exhausted"),
    ("Athlete12", "12846436186_Synchronflug"),
    ("Athlete12", "11962243206_Wundmanagement"),
    ("Athlete12", "13688068283_Luftpresser"),
    ("Athlete12", "11783093951_4x2000_sub8"),
    ("Athlete12", "13983533934_Technik__30er"),
    ("Athlete12", "11846980624_Platt_"),
    ("Athlete12", "14125110656_Vallah_isch_balla"),
]
# Extract test session data in the same format
test_session_dict = {k: v for k, v in hr_session_data.items() if k in test_sessions}

# Create a filtered hr_session_data for training only
train_session_dict = {k: v for k, v in hr_session_data.items() if k not in test_sessions}

In [125]:
len(train_session_dict)
from collections import Counter

# Map each athlete to their sport
athlete_sports = {athlete: meta["sport"] for athlete, meta in athletes.items()}

# Count sports in training sessions
sport_counts = Counter(athlete_sports[athlete] for (athlete, _) in train_session_dict.keys())

print("🚴‍♂️ Cycling sessions:", sport_counts.get("biking", 0))
print("🚣‍♀️ Rowing sessions:", sport_counts.get("rowing", 0))


🚴‍♂️ Cycling sessions: 8
🚣‍♀️ Rowing sessions: 52


In [126]:
# Base features
base_features = ['hr_roll_avg', 'hr_derivative', 'hr_second_derivative', 'hr_derivative_10s']

# Optional features to include only if present in ALL sessions
optional_features = ['hr_zone_hr_reserve', 'hr_zone_olympia']

# Start from base
feature_cols = base_features.copy()

# Check presence of optional features across all sessions
for feature in optional_features:
    if all(feature in df.columns for df in train_session_dict.values()):
        feature_cols.append(feature)

print("Using feature columns:", feature_cols)

Using feature columns: ['hr_roll_avg', 'hr_derivative', 'hr_second_derivative', 'hr_derivative_10s', 'hr_zone_olympia']


In [127]:
def create_cnn_training_data(
    train_session_dict,
    feature_cols,
    window_size=30,
    stride=1
):
    """
    Create CNN-ready training data from multiple sessions (optimized version).

    Parameters:
        train_session_dict (dict): {session_name: DataFrame}
        feature_cols (list): Columns to include in each window
        window_size (int): Length of each sequence (e.g., 30 seconds)
        stride (int): Steps to move the window
        label_strategy (str): 'any', 'last', or 'majority'

    Returns:
        X (np.ndarray): [num_windows, window_size, num_features]
        y (np.ndarray): [num_windows]
    """
    X = []
    y = []

    for session_name, df in train_session_dict.items():
        df = df.dropna(subset=feature_cols + ['manual_timestamp_numerical'])

        # Precompute feature and label arrays
        feature_array = df[feature_cols].values
        label_array = df['manual_timestamp_numerical'].astype(int).values

        for start in range(0, len(df) - window_size + 1, stride):
            end = start + window_size
            X_window = feature_array[start:end]
            y_window = label_array[start:end]

            # Skip incomplete window (defensive, though it shouldn’t occur)
            if len(X_window) < window_size:
                continue

            label = int(np.any(y_window))

            X.append(X_window)
            y.append(label)

    # Final array conversion
    X = np.stack(X)  # faster and safer than np.array(list)
    y = np.array(y, dtype=np.int32)

    return X, y


In [128]:
X_cnn, y_cnn = create_cnn_training_data(
    train_session_dict,
    feature_cols=feature_cols,
    window_size=30,
    stride=1
)

In [129]:
class IntervalCNN(nn.Module):
    def __init__(self, input_channels, seq_len):
        super(IntervalCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        reduced_seq_len = seq_len // 4
        self.fc1 = nn.Linear(128 * reduced_seq_len, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # Shape: (batch, channels, seq_len)
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze(1)


class IntervalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_model(model, train_loader, epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

In [130]:
# Set seed
torch.manual_seed(42)

X_cnn = np.array(X_cnn, dtype=np.float32)
y_cnn = np.array(y_cnn, dtype=np.float32)

# Create dataset & loader
g = torch.Generator()
g.manual_seed(SEED)

dataset = IntervalDataset(X_cnn, y_cnn)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True, generator=g)

# Build model
seq_len = X_cnn.shape[1]
num_features = X_cnn.shape[2]
model = IntervalCNN(input_channels=num_features, seq_len=seq_len)

# Train model
train_model(model, train_loader, epochs=10)


Epoch 1/10 - Loss: 0.2519
Epoch 2/10 - Loss: 0.2241
Epoch 3/10 - Loss: 0.2145
Epoch 4/10 - Loss: 0.2073
Epoch 5/10 - Loss: 0.2018
Epoch 6/10 - Loss: 0.1980
Epoch 7/10 - Loss: 0.1941
Epoch 8/10 - Loss: 0.1904
Epoch 9/10 - Loss: 0.1872
Epoch 10/10 - Loss: 0.1834


In [131]:
def predict_starts_hr(model, df, feature_cols, window_size=30, stride=1, threshold=0.5):
    """
    Predict interval starts in a session based on heart rate features using a trained CNN model.

    Parameters:
        model (torch.nn.Module): Trained PyTorch model
        df (pd.DataFrame): Session data
        feature_cols (list): List of features to use
        window_size (int): Size of sliding window (in timesteps)
        stride (int): Step between windows
        threshold (float): Threshold for converting probabilities to binary (0/1)

    Returns:
        preds_full (np.array): Binary array aligned to df (1 = predicted start)
        df_valid (pd.DataFrame): Cleaned DataFrame used for prediction
    """
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Drop rows with missing values in feature columns
    df_valid = df.dropna(subset=feature_cols).reset_index(drop=True)

    X = df_valid[feature_cols].values
    preds_full = np.zeros(len(df_valid), dtype=int)

    with torch.no_grad():
        for start in range(0, len(X) - window_size + 1, stride):
            end = start + window_size
            window = X[start:end]

            if window.shape[0] < window_size:
                continue

            window = window.astype(np.float32)
            input_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)  # Shape: (1, win, feat)
            output = model(input_tensor)
            prob = output.item()

            if prob >= threshold:
                preds_full[start] = 1  # Predict start at beginning of window

    return preds_full, df_valid

In [132]:
def predict_starts_hr_grouped(
    model,
    df,
    feature_cols,
    threshold=0.5,
    window_size=30,
    stride=1,
    gap_tolerance=1,
    group_strategy='directional',  # options: 'start', 'end', 'middle', 'directional'
):
    """
    Run CNN model on HR data and group nearby predicted interval starts.
    Allows for gaps of up to `gap_tolerance` seconds between groups.
    Chooses group point by strategy: 'start', 'end', 'middle', or 'directional'.

    Parameters:
        group_strategy (str): Strategy to mark the grouped prediction.
            'start' – first index,
            'end' – last index,
            'middle' – center index of the group,
            'directional' – based on average hr_derivative direction.

    Returns:
        preds_grouped (np.array): Binary predictions with grouped starts
        df_clean (pd.DataFrame): Aligned DataFrame after dropna and processing
    """
    model.eval()

    # Get raw predictions (probabilities) and aligned DataFrame
    preds_probs, df_clean = predict_starts_hr(
        model, df, feature_cols, window_size=window_size, stride=stride
    )
    preds_binary = (preds_probs > threshold).astype(int)

    # --- Smooth short gaps ---
    smoothed = preds_binary.copy()
    i = 0
    while i < len(smoothed) - gap_tolerance - 1:
        if smoothed[i] == 1:
            gap_range = smoothed[i + 1: i + 1 + gap_tolerance]
            after_gap = i + 1 + gap_tolerance
            if smoothed[after_gap] == 1 and all(gap_range == 0):
                smoothed[i + 1: after_gap] = 1
                i = after_gap
            else:
                i += 1
        else:
            i += 1

    # --- Grouping ---
    preds_grouped = np.zeros_like(smoothed)
    in_group = False
    group_start = None

    for i in range(len(smoothed)):
        if smoothed[i] == 1:
            if not in_group:
                in_group = True
                group_start = i
        else:
            if in_group:
                group_end = i - 1
                group_slice = slice(group_start, group_end + 1)

                # Select index based on strategy
                if group_strategy == 'start':
                    idx = group_start
                elif group_strategy == 'end':
                    idx = group_end
                elif group_strategy == 'middle':
                    idx = (group_start + group_end) // 2
                elif group_strategy == 'directional':
                    direction = df_clean.loc[group_slice, 'hr_derivative'].mean()
                    idx = group_start if direction >= 0 else group_end
                else:
                    raise ValueError(f"Unknown group_strategy: {group_strategy}")

                preds_grouped[idx] = 1
                in_group = False

    # Final group at end
    if in_group:
        group_end = len(smoothed) - 1
        group_slice = slice(group_start, group_end + 1)

        if group_strategy == 'start':
            idx = group_start
        elif group_strategy == 'end':
            idx = group_end
        elif group_strategy == 'middle':
            idx = (group_start + group_end) // 2
        elif group_strategy == 'directional':
            direction = df_clean.loc[group_slice, 'hr_derivative'].mean()
            idx = group_start if direction >= 0 else group_end
        else:
            raise ValueError(f"Unknown group_strategy: {group_strategy}")

        preds_grouped[idx] = 1

    return preds_grouped, df_clean


In [133]:
def evaluate_predictions(predictions, df_clean, beta=1.5, tolerance=15):
    """
    Computes precision, recall, Fβ against manual_timestamp_numerical ground truth,
    allowing ±tolerance seconds for matching a predicted start to a true start.

    Ignores NaNs (non-evaluated indices).
    """
    gt_array = df_clean['manual_timestamp_numerical'].fillna(0).astype(int).values
    valid_idx = ~np.isnan(predictions)

    preds = predictions[valid_idx]
    gt = gt_array[valid_idx]

    pred_indices = np.where(preds == 1)[0]
    true_indices = np.where(gt == 1)[0]

    # Match predicted starts to true starts within ±tolerance
    matched_true = set()
    matched_pred = set()

    for t in true_indices:
        for p in pred_indices:
            if abs(p - t) <= tolerance and p not in matched_pred:
                matched_true.add(t)
                matched_pred.add(p)
                break  # one prediction can only match one true

    tp = len(matched_true)
    fp = len(pred_indices) - len(matched_pred)
    fn = len(true_indices) - len(matched_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fbeta = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall) if (precision + recall) > 0 else 0.0

    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F{beta:.1f}-Score: {fbeta:.4f}")
    return precision, recall, fbeta

In [134]:
results = []

for session_key, df in test_session_dict.items():
    # Predict and group starts
    preds_grouped, df_clean = predict_starts_hr_grouped(
        model, df, feature_cols, window_size=30, stride=1, threshold=0.5, gap_tolerance=1, group_strategy='middle'
    )

    # Evaluate grouped predictions
    precision, recall, fbeta = evaluate_predictions(
        preds_grouped, df_clean, beta=1.5, tolerance=15
    )

    # Store results
    results.append({
        'Session': session_key if isinstance(session_key, str) else str(session_key),
        'Precision': precision,
        'Recall': recall,
        'Fβ-Score': fbeta
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Compute mean metrics
mean_metrics = results_df[['Precision', 'Recall', 'Fβ-Score']].mean()

# Output
print("📊 Mean metrics across test sessions:")
print(mean_metrics)


Precision: 0.7778, Recall: 0.7778, F1.5-Score: 0.7778
Precision: 0.2500, Recall: 0.3750, F1.5-Score: 0.3250
Precision: 0.3448, Recall: 1.0000, F1.5-Score: 0.6311
Precision: 0.2500, Recall: 1.0000, F1.5-Score: 0.5200
Precision: 0.2222, Recall: 0.5000, F1.5-Score: 0.3611
Precision: 0.2963, Recall: 0.8000, F1.5-Score: 0.5253
Precision: 0.1200, Recall: 0.3000, F1.5-Score: 0.2053
Precision: 0.7778, Recall: 0.8750, F1.5-Score: 0.8426
Precision: 0.3500, Recall: 0.6364, F1.5-Score: 0.5084
Precision: 0.1600, Recall: 0.2105, F1.5-Score: 0.1919
Precision: 0.5000, Recall: 0.4615, F1.5-Score: 0.4727
📊 Mean metrics across test sessions:
Precision    0.368082
Recall       0.630564
Fβ-Score     0.487369
dtype: float64


In [135]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Define zone colors (same order as original)
zone_labels_ordered = ["Zone1", "Zone2", "Zone3", "Zone4", "Zone5", "Zone6", "Zone7"]
base_colors = [
    "#00008B", "#5dade2", "#229954", "#f1c40f",
    "#e67e22", "#e74c3c", "#7b241c"
]
interval_colors = {label: color for label, color in zip(zone_labels_ordered, base_colors)}
interval_colors["Unclassified"] = "gray"


def plot_hr_with_intervals(df, preds, session_key=None, show=True, save_path=None):
    """
    Plot heart rate with predicted starts and ground truth intervals shaded using gt_zone_type.
    Also overlays power on a secondary y-axis for context.

    Parameters:
        df (pd.DataFrame): Must contain 'hr_roll_avg', 'power_roll_avg', 'timestamp', 'manual_timestamp_numerical', 'gt_zone_type'.
        preds (np.array): Binary predictions aligned with df.
        session_key (str or tuple): Optional identifier for plot title.
        show (bool): Whether to display the plot.
        save_path (str): If provided, saves the plot to this path.
    """
    # Ensure timestamp column exists for plotting
    if 'timestamp' not in df.columns:
        df = df.reset_index()
        if 'timestamp' not in df.columns:
            # Try to rename the index column if it's not yet named
            if 'index' in df.columns:
                df = df.rename(columns={'index': 'timestamp'})
            else:
                raise KeyError("Could not recover 'timestamp' column from index.")

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    time = df['timestamp'].reset_index(drop=True)
    hr = df['hr_roll_avg'].reset_index(drop=True)
    power = df['power_roll_avg'].reset_index(drop=True)

    fig, ax1 = plt.subplots(figsize=(15, 5))

    # Plot HR
    ax1.plot(time, hr, label='HR (Rolling Avg)', color='green', linewidth=1.5)

    # Ground truth zone shading
    if 'gt_zone_type' in df.columns and 'timestamp' in df.columns:
        valid = df['gt_zone_type'].notna() & (df['gt_zone_type'] != "Unclassified")
        df_valid = df[valid].copy()
        if not df_valid.empty:
            df_valid["gt_group"] = (df_valid["gt_zone_type"] != df_valid["gt_zone_type"].shift()).cumsum()
            grouped = df_valid.groupby("gt_group")

            for _, group in grouped:
                zone = group["gt_zone_type"].iloc[0]
                color = interval_colors.get(zone, "gray")
                ax1.axvspan(group["timestamp"].iloc[0], group["timestamp"].iloc[-1], color=color, alpha=0.2)

    # Predicted starts (on HR axis)
    pred_indices = np.where(preds == 1)[0]
    ax1.plot(time.iloc[pred_indices], hr.iloc[pred_indices], 'o',
             color='orange', label='Predicted Start', markersize=8)

    ax1.set_ylabel("Heart Rate (bpm)", color='green')
    ax1.tick_params(axis='y', labelcolor='green')

    # Power on secondary y-axis
    ax2 = ax1.twinx()
    ax2.plot(time, power, label='Power (Rolling Avg)', color='dimgrey', linestyle='-', linewidth=1.2)
    ax2.set_ylabel("Power (Watts)", color='dimgrey')
    ax2.tick_params(axis='y', labelcolor='dimgrey')

    # Title
    title = "Session HR + Power + Predicted Starts"
    if session_key:
        title = f"{session_key[0]} - {session_key[1]}"
    ax1.set_title(title)

    ax1.set_xlabel("Time")
    ax1.grid(True)

    # Legend: includes shaded zones and predicted start
    legend_elements = [Patch(facecolor=interval_colors[z], label=z, alpha=0.3) for z in zone_labels_ordered]
    legend_elements.append(Patch(facecolor='orange', label='Predicted Start'))

    ax1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)

    # Layout
    plt.subplots_adjust(bottom=0.25)

    if save_path:
        plt.savefig(save_path)
    if show:
        plt.show()
    plt.close(fig)


In [136]:
import os

# Directory for saving HR plots
output_dir = "Plots for Final Presentation/CNN HR"
os.makedirs(output_dir, exist_ok=True)

for session_key, df in test_session_dict.items():
    preds, df_clean = predict_starts_hr_grouped(model, df, feature_cols, threshold=0.5,
                                                window_size=30, stride=1, gap_tolerance=1, group_strategy='middle')

    athlete, session_name = session_key
    filename = f"{athlete}_{session_name}.png".replace(" ", "_")
    save_path = os.path.join(output_dir, filename)

    plot_hr_with_intervals(df_clean, preds, session_key=session_key, show=False, save_path=save_path)


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
import pandas as pd

# Zone colors (keep your palette)
zone_labels_ordered = ["Zone1", "Zone2", "Zone3", "Zone4", "Zone5", "Zone6", "Zone7"]
base_colors = [
    "#00008B", "#5dade2", "#229954", "#f1c40f",
    "#e67e22", "#e74c3c", "#7b241c"
]
interval_colors = {label: color for label, color in zip(zone_labels_ordered, base_colors)}
interval_colors["Unclassified"] = "gray"


def plot_hr_with_predictions(df, probs=None, pred_indices=None, gt_indices=None,
                             session_key=None, show=True, save_path=None,
                             show_probs=True):
    """
    Visualize HR, ground-truth zones, and predicted change points.

    Args:
        df (pd.DataFrame): must have ['timestamp', 'hr_roll_avg', 'power_roll_avg', 'gt_zone_type'].
        probs (np.ndarray, optional): model output probabilities, same length as df.
        pred_indices (list[int]): indices of predicted change points.
        gt_indices (list[int]): indices of ground-truth change points (manual labels == 1).
        session_key (tuple or str): (athlete, session_name)
        show (bool): whether to display the figure.
        save_path (str): optional path to save.
        show_probs (bool): overlay the model probabilities as a line.
    """
    if 'timestamp' not in df.columns:
        df = df.reset_index()
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    time = df['timestamp']
    hr = df['hr_roll_avg']
    power = df['power_roll_avg']

    fig, ax1 = plt.subplots(figsize=(15, 5))

    # Heart rate
    ax1.plot(time, hr, label='HR (Rolling Avg)', color='green', linewidth=1.5)

    # Optional: model probabilities
    if show_probs and probs is not None:
        ax1_twin = ax1.twinx()
        ax1_twin.plot(time, probs, color='black', alpha=0.4, linewidth=1.0, label='Model Probability')
        ax1_twin.set_ylabel("Pred. Probability", color='black', fontsize=9)
        ax1_twin.tick_params(axis='y', labelcolor='black')

    # Ground truth zone shading
    if 'gt_zone_type' in df.columns:
        valid = df['gt_zone_type'].notna() & (df['gt_zone_type'] != "Unclassified")
        df_valid = df[valid].copy()
        if not df_valid.empty:
            df_valid["gt_group"] = (df_valid["gt_zone_type"] != df_valid["gt_zone_type"].shift()).cumsum()
            for _, group in df_valid.groupby("gt_group"):
                zone = group["gt_zone_type"].iloc[0]
                color = interval_colors.get(zone, "gray")
                ax1.axvspan(group["timestamp"].iloc[0], group["timestamp"].iloc[-1],
                            color=color, alpha=0.2)

    # Ground-truth change points
    if gt_indices is not None and len(gt_indices) > 0:
        ax1.scatter(time.iloc[gt_indices], hr.iloc[gt_indices],
                    color='blue', marker='x', s=60, label='GT Start')

    # Predicted change points (top-k)
    if pred_indices is not None and len(pred_indices) > 0:
        ax1.scatter(time.iloc[pred_indices], hr.iloc[pred_indices],
                    color='orange', marker='o', s=70, label='Predicted Start')

    # Power on secondary y-axis
    ax2 = ax1.twinx()
    ax2.plot(time, power, label='Power (Rolling Avg)', color='dimgrey',
             linestyle='-', linewidth=1.2, alpha=0.8)
    ax2.set_ylabel("Power (W)", color='dimgrey')
    ax2.tick_params(axis='y', labelcolor='dimgrey')

    # Title
    title = "Session HR + Power + Change Points"
    if session_key is not None:
        title = f"{session_key[0]} - {session_key[1]}"
    ax1.set_title(title, fontsize=12)

    ax1.set_xlabel("Time")
    ax1.set_ylabel("Heart Rate (bpm)", color='green')
    ax1.tick_params(axis='y', labelcolor='green')
    ax1.grid(True, alpha=0.3)

    # Custom legend
    legend_elements = [
        Patch(facecolor=interval_colors[z], label=z, alpha=0.3)
        for z in zone_labels_ordered
    ]
    legend_elements += [
        Patch(facecolor='none', edgecolor='blue', label='GT Start'),
        Patch(facecolor='none', edgecolor='orange', label='Predicted Start')
    ]
    ax1.legend(handles=legend_elements, loc='upper center',
               bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=8)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    if show:
        plt.show()
    plt.close(fig)


NameError: name 'test_session_dict' is not defined